# Data Description
- Author: 詹明翰、謝子詮、曾翊涵（沿用已確認共同作者，不推定個別分工）
- Date first created: 20261126
- Background: Unit 1.3依人口及固定里界、官方碼表查證建立正式行政里crosswalk；這是代碼對照資料，不是年度人口特徵或事故歸里結果。
- Unit of observation: 一個固定研究行政里，一列一對人口碼與里界碼，共456列。
- Sample period: 用於2024–2025研究人口代碼連結至現有456里固定幾何；不宣稱歷史幾何完全一致。
- Known issues: 455精確區里名配對不等於官方跨碼認證；糖廍里1筆高可信暫定，保留來源糖?／糖蔀兩名稱；正式整合必須檢查verification_status，暫定列不得默認通過。
- Definition for each variable: 下表逐欄定義；11欄全部字串。

| Variable | Definition |
| --- | --- |
| village_id | 字串；本研究共同里鍵，等於boundary_code；唯一非空10碼CPTVID，不是全國通用轉碼宣告。 |
| population_code | 字串；人口CSV原始11碼區域代碼，保留原值，官方村里表另行核對。 |
| boundary_code | 字串；里界DBF原始10碼CPTVID，保留原值，不以整數表示。 |
| district_name | 字串；人口8碼所屬區碼經原CSV區列及官方RSCD0103名稱核對，與里界TNAME一致。 |
| population_village_name | 字串；人口CSV原始里名，糖?里保持問號，不修改來源。 |
| boundary_village_name | 字串；里界DBF原始VNAME，糖蔀里原字串保留。 |
| standard_village_name | 字串；官方村里代碼表所列名稱，包含糖廍里；不表示兩端幾何已獲認證。 |
| match_method | 字串；district_and_village_exact為區里名精確配對；official_population_code_plus_empirical_local_rule為官方人口碼旁證＋此批資料規律候選。 |
| verification_status | 字串；verified_official／verified_exact_match／provisional_match／unmatched；本檔實際455筆verified_exact_match、1筆provisional_match，其餘0。 |
| source_evidence | 字串；精確來源路徑、hash、ZIP member、官方歷史索引網址及處理Notebook；來源包保存原始回應。 |
| notes | 字串；配對依據、官方跨碼證據不足及固定幾何限制；暫定列不可默認用於正式整合。 |

- Data path: `data/processed/village_crosswalk.csv`；[實際CSV](village_crosswalk.csv)
- Source data: 人口CSV、五個里界配套及[官方碼表來源包](../raw/unit13_official_code_evidence.zip)，精確路徑及固定hash列於執行程式。
- Processing code: [人口驗證及crosswalk](../code/taipei_population_validation.ipynb)
- Role: `crosswalk`（非analysis／train／test人口資料）
- Verification status: 以實際執行的獨立重新讀檔、唯一性／來源一致性斷言為準；保留暫定狀態。
- SHA-256: `dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69`
- Downstream analysis input: `data/processed/village_crosswalk.csv`；僅供後續明確檢查配對狀態後整合，不直接作分群矩陣。

來源文件：[人口raw](../raw/taipei_population_age_raw_documentation.ipynb)、[里界raw](../raw/taipei_village_boundaries_raw_documentation.ipynb)、[官方證據包說明](../raw/unit13_official_code_evidence_documentation.ipynb)。


In [1]:
from pathlib import Path
import hashlib,json,sys,platform,os,io,zipfile,re
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
def project_root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():return p
    raise FileNotFoundError('Open within the student project containing AGENTS.md and templates/.')
ROOT=project_root()
assert Path(sys.executable).resolve()==(ROOT/'.venv/Scripts/python.exe').resolve()
assert sys.prefix!=sys.base_prefix
os.environ['PROJ_NETWORK']='OFF'
def file(relative,area='data',must_exist=True):
    rel=Path(relative)
    if rel.is_absolute() or '..' in rel.parts:raise ValueError('Explicit project-relative path required')
    path=(ROOT/rel).resolve()
    if not path.is_relative_to((ROOT/area).resolve()):raise ValueError('Path outside intended project area')
    if must_exist and not path.is_file():raise FileNotFoundError(relative)
    return path
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for b in iter(lambda:stream.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def verify(expected):
    rows=[]
    for relative,digest in expected.items():
        path=file(relative)
        assert sha(path)==digest,f'Input changed; stop and inspect version: {relative}'
        rows.append({'path':relative,'bytes':path.stat().st_size,'sha256':digest})
    return pd.DataFrame(rows)
def show(title,table,block=20):
    display(Markdown('**'+title+'**'))
    if not len(table):display(Markdown('無符合紀錄。'))
    for start in range(0,len(table),block):display(table.iloc[start:start+block])
pd.set_option('display.max_colwidth',110)
print({'python':platform.python_version(),'interpreter_is_project_venv':True,'network_in_notebook':'none'})

EXPECTED_SHA256={'data/raw/112迄今各里人口數按年齡分.csv': 'a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641', 'data/raw/G97_A_CAVLGE_P.shp': 'f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc', 'data/raw/G97_A_CAVLGE_P.shx': 'ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e', 'data/raw/G97_A_CAVLGE_P.dbf': '2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577', 'data/raw/G97_A_CAVLGE_P.prj': '8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5', 'data/raw/G97_A_CAVLGE_P.cpg': '3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824', 'data/raw/taipei_population_age_raw_documentation.ipynb': '8b7b5376b6e79821c0cdaffd7f17e97f9e466bb3933d6b77552cf48846c29f53', 'data/raw/taipei_village_boundaries_raw_documentation.ipynb': '74c1c7e8fca3f5c5a82a94282f25dd42b6c2ba91e952e6408c3bd307fb269fcf', 'data/code/taipei_village_boundary_validation.ipynb': '662dbee2dc42c3552b12628a1e89fcc81b07155c225277117f50d07e0145a343', 'data/raw/unit13_official_code_evidence.zip': '9f27bd2f9211f4c21253876529efec1414b82f0b7014f6d4028b70f66b58cc40', 'data/raw/unit13_official_code_evidence_documentation.ipynb': '08f9972f534920668134f919c6f460c915918e92bd599031dcec61c006589682', 'data/code/taipei_population_validation.ipynb': 'e26ff652ccfa2113db041322a99a000d8cb6d71acccdab66c35448fcddaab7d8', 'data/processed/village_crosswalk.csv': 'dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69'}
show('逐檔資料／處理程式／來源hash',verify(EXPECTED_SHA256),block=6)


{'python': '3.12.2', 'interpreter_is_project_venv': True, 'network_in_notebook': 'none'}


**逐檔資料／處理程式／來源hash**

,path,bytes,sha256
0,data/raw/112迄今各里人口數按年齡分.csv,21284860,a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641
1,data/raw/G97_A_CAVLGE_P.shp,765784,f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc
2,data/raw/G97_A_CAVLGE_P.shx,3748,ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e
3,data/raw/G97_A_CAVLGE_P.dbf,272706,2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577
4,data/raw/G97_A_CAVLGE_P.prj,399,8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5
5,data/raw/G97_A_CAVLGE_P.cpg,5,3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824


,path,bytes,sha256
6,data/raw/taipei_population_age_raw_documentation.ipynb,284126,8b7b5376b6e79821c0cdaffd7f17e97f9e466bb3933d6b77552cf48846c29f53
7,data/raw/taipei_village_boundaries_raw_documentation.ipynb,106813,74c1c7e8fca3f5c5a82a94282f25dd42b6c2ba91e952e6408c3bd307fb269fcf
8,data/code/taipei_village_boundary_validation.ipynb,1802468,662dbee2dc42c3552b12628a1e89fcc81b07155c225277117f50d07e0145a343
9,data/raw/unit13_official_code_evidence.zip,300990,9f27bd2f9211f4c21253876529efec1414b82f0b7014f6d4028b70f66b58cc40
10,data/raw/unit13_official_code_evidence_documentation.ipynb,55667,08f9972f534920668134f919c6f460c915918e92bd599031dcec61c006589682
11,data/code/taipei_population_validation.ipynb,142657,e26ff652ccfa2113db041322a99a000d8cb6d71acccdab66c35448fcddaab7d8


,path,bytes,sha256
12,data/processed/village_crosswalk.csv,362330,dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69


## Change for Your Own Data
固定讀取本檔UTF-8-SIG CSV；所有欄指定字串，不套預設NA字串或數值推斷。來源或本檔hash變動即停止。


In [2]:
df=pd.read_csv(file('data/processed/village_crosswalk.csv'),dtype='string',encoding='utf-8-sig',keep_default_na=False)
COLUMNS=['village_id','population_code','boundary_code','district_name','population_village_name','boundary_village_name',
         'standard_village_name','match_method','verification_status','source_evidence','notes']
assert list(df.columns)==COLUMNS
print({'path':'data/processed/village_crosswalk.csv','rows':len(df),'columns':len(df.columns),'role':'crosswalk',
       'read_dtype':'all string','saved_sha256':sha(file('data/processed/village_crosswalk.csv'))})
show('逐欄型態／非空／唯一值',pd.DataFrame({'dtype':df.dtypes.astype(str),'missing':df.isna().sum(),
                                         'blank':df.eq('').sum(),'unique_values':df.nunique()}))


{'path': 'data/processed/village_crosswalk.csv', 'rows': 456, 'columns': 11, 'role': 'crosswalk', 'read_dtype': 'all string', 'saved_sha256': 'dd9e762419c302cf5b4bb19daad2673d7d54d0bb750204e7e33ecbf1a9983f69'}


**逐欄型態／非空／唯一值**

,dtype,missing,blank,unique_values
village_id,string,0,0,456
population_code,string,0,0,456
boundary_code,string,0,0,456
district_name,string,0,0,12
population_village_name,string,0,0,454
boundary_village_name,string,0,0,454
standard_village_name,string,0,0,454
match_method,string,0,0,2
verification_status,string,0,0,2
source_evidence,string,0,0,1


## Summary
### Sample
實際CSV前十列，依欄分組呈現，不替換成抽樣或排序後紀錄。


In [3]:
show('前十列：識別與配對欄',df.iloc[:10,:9],block=10)
show('前十列：來源與限制',df.iloc[:10,[0,9,10]],block=10)


**前十列：識別與配對欄**

,village_id,population_code,boundary_code,district_name,population_village_name,boundary_village_name,standard_village_name,match_method,verification_status
0,6300100002,63000010002,6300100002,松山區,莊敬里,莊敬里,莊敬里,district_and_village_exact,verified_exact_match
1,6300100003,63000010003,6300100003,松山區,東榮里,東榮里,東榮里,district_and_village_exact,verified_exact_match
2,6300100004,63000010004,6300100004,松山區,三民里,三民里,三民里,district_and_village_exact,verified_exact_match
3,6300100005,63000010005,6300100005,松山區,新益里,新益里,新益里,district_and_village_exact,verified_exact_match
4,6300100006,63000010006,6300100006,松山區,富錦里,富錦里,富錦里,district_and_village_exact,verified_exact_match
5,6300100007,63000010007,6300100007,松山區,新東里,新東里,新東里,district_and_village_exact,verified_exact_match
6,6300100008,63000010008,6300100008,松山區,富泰里,富泰里,富泰里,district_and_village_exact,verified_exact_match
7,6300100009,63000010009,6300100009,松山區,介壽里,介壽里,介壽里,district_and_village_exact,verified_exact_match
8,6300100010,63000010010,6300100010,松山區,精忠里,精忠里,精忠里,district_and_village_exact,verified_exact_match
9,6300100011,63000010011,6300100011,松山區,東光里,東光里,東光里,district_and_village_exact,verified_exact_match


**前十列：來源與限制**

,village_id,source_evidence,notes
0,6300100002,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
1,6300100003,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
2,6300100004,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
3,6300100005,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
4,6300100006,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
5,6300100007,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
6,6300100008,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
7,6300100009,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
8,6300100010,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。
9,6300100011,data/raw/112迄今各里人口數按年齡分.csv@a663a4f0329b18bb0514da7fd9f6845312d08fadef71e566faf1930c03541641;data/raw/G97_...,行政區及原始里名精確一對一；官方人口名稱碼表旁證，不代表官方跨碼認證或歷史幾何相同。


### Summary Stats
本檔無連續人口變數；以配對狀態、行政區及來源名稱差異等適用摘要取代數值平均。


In [4]:
show('四種配對狀態完整列示',df.verification_status.value_counts().reindex(
     ['verified_official','verified_exact_match','provisional_match','unmatched'],fill_value=0).rename('villages').to_frame())
show('各區配對狀態',pd.crosstab(df.district_name,df.verification_status))
show('全部暫定／未配對列',df.loc[~df.verification_status.isin(['verified_official','verified_exact_match'])].drop(columns='source_evidence'))
print({'pairing_coverage_including_provisional':len(df)/456,'verified_exact_fraction':int(df.verification_status.eq('verified_exact_match').sum())/456,
       'provisional_fraction':int(df.verification_status.eq('provisional_match').sum())/456,
       'population_name_alone_is_unique_citywide':bool(df.population_village_name.is_unique)})


**四種配對狀態完整列示**

,villages
verification_status,
verified_official,0
verified_exact_match,455
provisional_match,1
unmatched,0


**各區配對狀態**

verification_status,provisional_match,verified_exact_match
district_name,,
中山區,0,42
中正區,0,31
信義區,0,41
內湖區,0,39
北投區,0,42
南港區,0,20
士林區,0,51
大同區,0,25
大安區,0,53


**全部暫定／未配對列**

,village_id,population_code,boundary_code,district_name,population_village_name,boundary_village_name,standard_village_name,match_method,verification_status,notes
239,6300700015,63000070015,6300700015,萬華區,糖?里,糖蔀里,糖廍里,official_population_code_plus_empirical_local_rule,provisional_match,高可信候選：官方五版本均為糖廍里63000070015；本機糖蔀里6300700015符合455里規律，但缺官方跨碼／名稱誤植更正證明。正式整合須明確審查狀態。


{'pairing_coverage_including_provisional': 1.0, 'verified_exact_fraction': 0.9978070175438597, 'provisional_fraction': 0.0021929824561403508, 'population_name_alone_is_unique_citywide': False}


### Assertions
重新讀原始來源的必要欄位及官方碼表，獨立檢查一對一關係、原名保留及標準名稱。
下游必須明列可用狀態，保留未通過者及覆蓋率；455筆可驗證配對與1筆待審不互相混稱。


In [5]:
assert len(df)==456 and not df.isna().to_numpy().any() and not df.eq('').to_numpy().any()
assert df[['village_id','population_code','boundary_code']].nunique().eq(456).all()
assert df.village_id.eq(df.boundary_code).all()
assert df.population_code.str.fullmatch(r'\d{11}').all() and df.boundary_code.str.fullmatch(r'\d{10}').all()
assert df.verification_status.value_counts().to_dict()=={'verified_exact_match':455,'provisional_match':1}
source=pd.read_csv(file('data/raw/112迄今各里人口數按年齡分.csv'),encoding='cp950',dtype='string',keep_default_na=False,
                 usecols=['年份','區域代碼','區域別','性別'])
source=source.loc[source['年份'].isin(['113','114']) & source['區域代碼'].str.len().eq(11)]
for year in ['113','114']:assert set(source.loc[source['年份'].eq(year),'區域代碼'])==set(df.population_code)
source_names=source[['區域代碼','區域別']].drop_duplicates().set_index('區域代碼')['區域別']
assert df.population_code.map(source_names).eq(df.population_village_name).all()
import geopandas as gpd
attributes=gpd.read_file(file('data/raw/G97_A_CAVLGE_P.shp'),encoding='utf-8',engine='pyogrio',ignore_geometry=True).set_index('CPTVID')
assert set(attributes.index)==set(df.boundary_code)
assert df.boundary_code.map(attributes.VNAME).eq(df.boundary_village_name).all()
assert df.boundary_code.map(attributes.TNAME).eq(df.district_name).all()
zip_source=zipfile.ZipFile(file('data/raw/unit13_official_code_evidence.zip'))
official=pd.read_csv(io.BytesIO(zip_source.read('11509VillageUTF8.txt')),header=None,dtype='string',encoding='utf-8-sig',
                     names=['code','district','name'],keep_default_na=False).set_index('code')
assert df.population_code.map(official['name']).eq(df.standard_village_name).all()
exact=df.verification_status.eq('verified_exact_match')
assert df.loc[exact,'population_village_name'].eq(df.loc[exact,'boundary_village_name']).all()
exceptional=df.loc[~exact]
assert len(exceptional)==1 and exceptional.population_code.iloc[0]=='63000070015'
assert exceptional.boundary_code.iloc[0]=='6300700015' and exceptional.standard_village_name.iloc[0]=='糖廍里'
assert exceptional.verification_status.iloc[0]=='provisional_match'
assert df.source_evidence.str.contains(EXPECTED_SHA256['data/raw/unit13_official_code_evidence.zip'],regex=False).all()
verify(EXPECTED_SHA256)
print({'serialized_crosswalk_verified':True,'source_original_names_preserved':True,'independent_source_checks_passed':True,
       'verified_for_explicit_status_gated_integration':455,'pending_provisional_review':1,'annual_population_dataset_created':False})


{'serialized_crosswalk_verified': True, 'source_original_names_preserved': True, 'independent_source_checks_passed': True, 'verified_for_explicit_status_gated_integration': 455, 'pending_provisional_review': 1, 'annual_population_dataset_created': False}
